*This code rotates Quadrupolar tensor from PAS (experimental) to tenon frame and Crystal frame (quantum computed)to tenon frame*

*Shiva Agarwal*
*20 January 2025*

In [1]:
import numpy as np

In [5]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [26]:
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    print('Direction Cosine Matrix: \n', dc, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [20]:
def rotate_V(cq, etaq, a, b, g):
    V_PAS = np.zeros((3,3))
    V_PAS[0,0] = -(1 + etaq) * cq/2
    V_PAS[1,1] = -(1 - etaq) * cq/2
    V_PAS[2,2] = cq
    

    U = Rabc(a, b, g)
    V_T = np.matmul(np.matmul(U, V_PAS), np.linalg.inv(U))

    print('Quadrupolar tensor in PAS: \n',V_PAS)
    print('Quadrupolar tensor in Tenon (experimental): \n',V_T)
    
    return V_T

**LHQipc2B**

In [3]:
efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]=-0.0285; efg[0,1]=0.1852; efg[0,2]=0.1112;
efg[1,0]=efg[0,1]; efg[1,1]=-0.0059; efg[1,2]=0.0266;
efg[2,0]=efg[0,2]; efg[2,1]=efg[1,2]; efg[2,2]=0.0344;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.04059
V_LHQ_crystal = efg*Q*234.9647

print(V_LHQ_crystal)

[[-0.27181069  1.76629262  1.06053855]
 [ 1.76629262 -0.05626958  0.25368998]
 [ 1.06053855  0.25368998  0.32808027]]


In [16]:
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma) 


V_LHQ_tenon = np.matmul(np.matmul(np.linalg.inv(U), V_LHQ_crystal), U)



print('==========================')
print('\n Quad Tensor in Crystal Frame: \n', V_LHQ_crystal)  #from magres
print('\n Quad Tensor in Tenon Frame (Calculated): \n', V_LHQ_tenon)


 Quad Tensor in Crystal Frame: 
 [[-0.27181069  1.76629262  1.06053855]
 [ 1.76629262 -0.05626958  0.25368998]
 [ 1.06053855  0.25368998  0.32808027]]

 Quad Tensor in Tenon Frame (Calculated): 
 [[ 0.2801114  -0.83813975 -1.35455866]
 [-0.83813975 -1.36878474 -0.54341786]
 [-1.35455866 -0.54341786  1.08867334]]


In [22]:
# Find Quad tensor in PAS

# Find Quadrupolar Tensor and rotate in tenon frame
# Values from ASICS
cq = 2.26
etaq = 0.78
psi = 84
chi = 89
xi = 272

V_LHQ_T = rotate_V(cq, etaq, psi, chi, xi)





Quadrupolar tensor in PAS: 
 [[-2.0114  0.      0.    ]
 [ 0.     -0.2486  0.    ]
 [ 0.      0.      2.26  ]]
Quadrupolar tensor in Tenon (experimental): 
 [[-1.98718433  0.14508985 -0.18465327]
 [ 0.14508985  2.25427508 -0.03768919]
 [-0.18465327 -0.03768919 -0.26709075]]


In [27]:
sort_eigenvalues(V_LHQ_T )

 Unsorted Eigenvalues:
 [-2.0114 -0.2486  2.26  ] 

 Unsorted Eigenvectors:
 [[-0.99397973  0.10385904  0.03489418]
 [ 0.03288515 -0.02099422  0.99923861]
 [-0.10451254 -0.99437042 -0.01745241]] 

Sorted Eigenvalues: 
 [-0.2486 -2.0114  2.26  ] 

Sorted Eigenvectors: 
 [[ 0.10385904 -0.99397973  0.03489418]
 [-0.02099422  0.03288515  0.99923861]
 [-0.99437042 -0.10451254 -0.01745241]] 

Direction Cosine Matrix: 
 [[-0.99397973  0.10385904  0.03489418]
 [ 0.03288515 -0.02099422  0.99923861]
 [-0.10451254 -0.99437042 -0.01745241]] 



(array([-0.2486, -2.0114,  2.26  ]),
 array([[-0.99397973,  0.10385904,  0.03489418],
        [ 0.03288515, -0.02099422,  0.99923861],
        [-0.10451254, -0.99437042, -0.01745241]]),
 0.0,
 array([-2.0114, -0.2486,  2.26  ]),
 array([[-0.99397973,  0.10385904,  0.03489418],
        [ 0.03288515, -0.02099422,  0.99923861],
        [-0.10451254, -0.99437042, -0.01745241]]))